## TorchAO CPU INT8 PTQ (Appendix)

Post-training quantization on CPU via TorchAO's `int8_x86_pt2e` path.
Calibrates on the holdout-val split, evaluates on the same 5k samples. Multi-seed.

In [ ]:
SKIP_EXISTING = True
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/cpu"

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [ ]:
import json
import time
import pandas as pd

from src.config import ExperimentConfig, with_overrides, set_seed
from src.runner import run_experiment

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)

In [ ]:
INPUT_BITS = [8, 4, 2, 1]
SEEDS = [1, 2, 42]

OUT_DIR = Path(OUTPUT_ROOT).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Run Inference

In [ ]:
all_records = []

for bits in INPUT_BITS:
    for seed in SEEDS:
        run_id = f"torchao_ptq_int8_b{bits}_cpu"
        result_path = OUT_DIR / run_id / "result.json"

        if SKIP_EXISTING and result_path.exists():
            print(f"  SKIP in{bits}b/seed_{seed}")
            with open(result_path) as f:
                all_records.append(json.load(f))
            continue

        print(f"\n--- CPU INT8 PTQ, input_bits={bits}, seed={seed} ---")
        cfg = ExperimentConfig(
            backend="torchao_cpu_ptq",
            device="cpu",
            batch_size=1,
            model_precision="int8",
            seed=seed,
            num_eval_batches=500,
            cpu_calib_num_batches=1,
            input_quant_bits=bits,
            output_root=str(OUT_DIR),
        )
        set_seed(cfg)

        t0 = time.perf_counter()
        payload, tracker = run_experiment(cfg, save_results_flag=False)
        elapsed = time.perf_counter() - t0

        r = payload["results"]
        print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.3f}ms")

        payload["config_extra"] = {
            "input_quant_bits": bits,
            "seed": seed,
            "backend": "torchao_cpu_ptq",
            "split": "holdout_val",
        }
        payload["total_eval_time_sec"] = round(elapsed, 3)

        result_path.parent.mkdir(parents=True, exist_ok=True)
        with open(result_path, "w") as f:
            json.dump(payload, f, indent=2, sort_keys=True)

        all_records.append(payload)

print(f"\n{len(all_records)} runs complete.")

## Per-Run Results

In [ ]:
rows = []
for rec in all_records:
    r = rec["results"]
    extra = rec.get("config_extra", rec.get("config", {}))
    bits = extra.get("input_quant_bits", rec["config"].get("input_quant_bits", None))
    seed = extra.get("seed", 42)
    rows.append({
        "method": "cpu_ptq_int8",
        "input_bits": bits,
        "seed": seed,
        "top1": r["top1_acc"],
        "top5": r["top5_acc"],
        "lat_ms": r["infer_ms_avg"],
        "lat_std": r.get("infer_ms_std", None),
        "throughput": r.get("throughput_infer_sps", None),
    })

df_all = pd.DataFrame(rows).sort_values(
    ["input_bits", "seed"], ascending=[True, True]
).reset_index(drop=True)
df_all

## Averaged Results (mean ± std across seeds)

In [ ]:
avg_df = df_all.groupby(["method", "input_bits"]).agg(
    top1_mean=("top1", "mean"),
    top1_std=("top1", "std"),
    top5_mean=("top5", "mean"),
    top5_std=("top5", "std"),
    infer_ms_mean=("lat_ms", "mean"),
    infer_ms_std=("lat_ms", "std"),
    throughput_mean=("throughput", "mean"),
    n_seeds=("seed", "count"),
).round(3).reset_index()

avg_df = avg_df.sort_values("input_bits").reset_index(drop=True)
avg_df

## Save Results

In [ ]:
out_path = PROJECT_ROOT / "resultsv2" / "appendix" / "cpu_ptq_int8_val_results.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")